In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda3/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("best_hp_reg.json") as f:
    hp = json.load(f)
print(f"Entries: {len(hp)}")  # should be 21
print(sorted(hp.keys()))

Entries: 21
['gatv2_pbsg', 'gatv2_smiles', 'gatv2_smiles_gl', 'gcn_pbsg', 'gcn_smiles', 'gcn_smiles_gl', 'gine_pbsg', 'gine_smiles', 'gine_smiles_gl', 'rf_fp_poly', 'rf_fp_pooled_poly', 'rf_fp_pooled_ru', 'rf_fp_ru', 'rr_fp_poly', 'rr_fp_pooled_poly', 'rr_fp_pooled_ru', 'rr_fp_ru', 'xgb_fp_poly', 'xgb_fp_pooled_poly', 'xgb_fp_pooled_ru', 'xgb_fp_ru']


In [3]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [6]:
with open("random_stratified_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=345  test=62  folds=5
Device: cpu


In [7]:
X_fp_RU_pooled = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly_pooled      = np.load("Vectors/X_fp_pooled_poly.npy")

y=np.load("Vectors/y.npy")

In [8]:
FP_NAMES     = [f"morgan_fp_{i}" for i in range(1024)]
DESC_NAMES   = ["MolWt","LogP","TPSA","RotBonds","Rings","HBD","HBA","Heteroatoms","NHOH",
                "AliphaticRings","AmideBonds","AromaticRings","backbone_atoms","sidechain_atoms",
                "avg_branch_length","flexibility","side_bulkiness_frac","kier_flex"]
ARCH_NAMES   = ["arch_alternating","arch_block","arch_random","arch_homopolymer"]
STEREO_NAMES = ["stereo_isotactic","stereo_syndiotactic","stereo_stereo-irregular",
                "stereo_atactic","stereo_achiral","stereo_unknown"]
SEQ_NAMES    = ["Pm","Px","asym"]
EDGE_NAMES   = ["edge_stereo"]
ALL_NAMES    = FP_NAMES + DESC_NAMES + ARCH_NAMES + STEREO_NAMES + SEQ_NAMES + EDGE_NAMES

print("total names:", len(ALL_NAMES))
assert len(ALL_NAMES) == 1056
assert X_fp_RU_pooled.shape[1] == len(ALL_NAMES)
print("names align with matrix width ✓")

total names: 1056
names align with matrix width ✓


In [9]:
print("RU  :", X_fp_RU_pooled.shape)      # expect (407, 1056)
print("poly:", X_fp_poly_pooled.shape)    # expect (407, 1056)
print("y   :", y.shape)                   # expect (407,)
assert X_fp_RU_pooled.shape[1] == len(ALL_NAMES) == 1056
assert X_fp_poly_pooled.shape[1] == 1056
assert X_fp_RU_pooled.shape[0] == len(y)

RU  : (407, 1056)
poly: (407, 1056)
y   : (407,)


In [10]:
print("FP block (cols 0-4, expect 0/1 or small avg):", X_fp_RU_pooled[0, :5])
print("DESC start (col 1024, expect MolWt ~tens-hundreds):", X_fp_RU_pooled[0, 1024:1027], "→", DESC_NAMES[:3])
print("ARCH block (cols 1042-1045, expect one-hot):", X_fp_RU_pooled[0, 1042:1046], "→", ARCH_NAMES)
print("STEREO block (cols 1046-1051, expect one-hot):", X_fp_RU_pooled[0, 1046:1052], "→", STEREO_NAMES)
print("SEQ block (cols 1052-1054, Pm/Px/asym in [0,1]):", X_fp_RU_pooled[0, 1052:1055], "→", SEQ_NAMES)
print("EDGE (col 1055, in [0,1]):", X_fp_RU_pooled[0, 1055], "→", EDGE_NAMES)

FP block (cols 0-4, expect 0/1 or small avg): [0. 1. 0. 0. 0.]
DESC start (col 1024, expect MolWt ~tens-hundreds): [98.10103     0.48789984 26.299994  ] → ['MolWt', 'LogP', 'TPSA']
ARCH block (cols 1042-1045, expect one-hot): [0. 0. 0. 1.] → ['arch_alternating', 'arch_block', 'arch_random', 'arch_homopolymer']
STEREO block (cols 1046-1051, expect one-hot): [0. 0. 1. 0. 0. 0.] → ['stereo_isotactic', 'stereo_syndiotactic', 'stereo_stereo-irregular', 'stereo_atactic', 'stereo_achiral', 'stereo_unknown']
SEQ block (cols 1052-1054, Pm/Px/asym in [0,1]): [0.33333334 0.         0.5       ] → ['Pm', 'Px', 'asym']
EDGE (col 1055, in [0,1]): 0.33333334 → ['edge_stereo']


In [11]:
STEREO_COLS = set(STEREO_NAMES + ["Pm"] + EDGE_NAMES)          # 6 + 1 + 1 = 8
keep_idx = [i for i, n in enumerate(ALL_NAMES) if n not in STEREO_COLS]
drop_idx = [i for i, n in enumerate(ALL_NAMES) if n in STEREO_COLS]

assert len(keep_idx) == 1048 and len(drop_idx) == 8, (len(keep_idx), len(drop_idx))
print("DROPPING:", [ALL_NAMES[i] for i in drop_idx])
print("kept SEQ (should include Px, asym):",
      [ALL_NAMES[i] for i in keep_idx if ALL_NAMES[i] in SEQ_NAMES])

DROPPING: ['stereo_isotactic', 'stereo_syndiotactic', 'stereo_stereo-irregular', 'stereo_atactic', 'stereo_achiral', 'stereo_unknown', 'Pm', 'edge_stereo']
kept SEQ (should include Px, asym): ['Px', 'asym']


In [12]:
import numpy as np
dropped = X_fp_RU_pooled[:, drop_idx]
print("dropped-col variances:", dropped.var(axis=0).round(4))
print("  names:", [ALL_NAMES[i] for i in drop_idx])

dropped-col variances: [0.2369 0.1342 0.0445 0.1078 0.0533 0.     0.2075 0.2075]
  names: ['stereo_isotactic', 'stereo_syndiotactic', 'stereo_stereo-irregular', 'stereo_atactic', 'stereo_achiral', 'stereo_unknown', 'Pm', 'edge_stereo']


In [13]:
def strip_stereo_pooled(X):
    assert X.shape[1] == 1056, X.shape
    return X[:, keep_idx]

X_ru_ab   = strip_stereo_pooled(X_fp_RU_pooled)      # (407, 1048)
X_poly_ab = strip_stereo_pooled(X_fp_poly_pooled)    # (407, 1048)

print("ablated RU  :", X_ru_ab.shape)
print("ablated poly:", X_poly_ab.shape)
# confirm it's genuinely different from full (not accidentally a view/no-op)
print("differs from full:", X_ru_ab.shape[1] != X_fp_RU_pooled.shape[1])

ablated RU  : (407, 1048)
ablated poly: (407, 1048)
differs from full: True


In [14]:
# candidate matched sets: identical KEPT (non-stereo) features, differing DROPPED (stereo) features
from collections import defaultdict
kept   = X_fp_RU_pooled[:, keep_idx]      # non-stereo part (1048)
groups = defaultdict(list)
for i in range(len(kept)):
    groups[tuple(kept[i].round(4))].append(i)   # same chemistry+arch+composition

matched = {k: v for k, v in groups.items() if len(v) > 1}   # ≥2 polymers, same non-stereo features
print(f"{len(matched)} matched-chemistry groups, "
      f"{sum(len(v) for v in matched.values())} polymers total")
for k, idxs in list(matched.items())[:5]:
    print("  group size", len(idxs), "→ rows", idxs,
          "| stereo classes:", [df.iloc[i]['stereo_class'] for i in idxs] if 'df' in dir() else idxs)

61 matched-chemistry groups, 243 polymers total
  group size 3 → rows [0, 1, 2] | stereo classes: ['stereo-irregular', 'stereo-irregular', 'isotactic']
  group size 9 → rows [4, 5, 6, 7, 8, 9, 10, 403, 404] | stereo classes: ['stereo-irregular', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic']
  group size 8 → rows [14, 15, 16, 17, 18, 19, 20, 21] | stereo classes: ['isotactic', 'stereo-irregular', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic']
  group size 2 → rows [22, 28] | stereo classes: ['isotactic', 'isotactic']
  group size 3 → rows [23, 24, 29] | stereo classes: ['isotactic', 'isotactic', 'isotactic']


In [15]:
# keep only groups that actually CONTRAST stereo (≥2 distinct stereo classes)
stereo_contrasting = {}
for k, idxs in matched.items():
    classes = set(df.iloc[i]['stereo_class'] for i in idxs)
    if len(classes) > 1:
        stereo_contrasting[k] = idxs

n_groups = len(stereo_contrasting)
n_polys  = sum(len(v) for v in stereo_contrasting.values())
print(f"{n_groups} stereo-contrasting groups, {n_polys} polymers")
for k, idxs in list(stereo_contrasting.items())[:8]:
    print("  size", len(idxs), "rows", idxs,
          "classes:", [df.iloc[i]['stereo_class'] for i in idxs])

15 stereo-contrasting groups, 102 polymers
  size 3 rows [0, 1, 2] classes: ['stereo-irregular', 'stereo-irregular', 'isotactic']
  size 9 rows [4, 5, 6, 7, 8, 9, 10, 403, 404] classes: ['stereo-irregular', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic']
  size 8 rows [14, 15, 16, 17, 18, 19, 20, 21] classes: ['isotactic', 'stereo-irregular', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic', 'isotactic']
  size 2 rows [34, 35] classes: ['isotactic', 'atactic']
  size 11 rows [36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46] classes: ['syndiotactic', 'stereo-irregular', 'stereo-irregular', 'stereo-irregular', 'syndiotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic', 'syndiotactic']
  size 5 rows [52, 53, 54, 55, 56] classes: ['syndiotactic', 'stereo-irregular', 'stereo-irregular', 'syndiotactic', 'syndiotactic']
  size 3 rows [72, 73, 74] classes: ['syndiotactic', 'syndiotact

In [16]:
import importlib
import scripts.test_eval_ablation_models_reg
importlib.reload(scripts.test_eval_ablation_models_reg)

<module 'scripts.test_eval_ablation_models_reg' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML/Section_S4/Regressor/scripts/test_eval_ablation_models_reg.py'>

In [17]:
from scripts.test_eval_ablation_models_reg import run_all_test_reg

test_results_reg = run_all_test_reg(
    X_ru_ab,  X_poly_ab,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_reg.json",
)

Loaded HPs from best_hp_reg.json  (21 entries)

── XGB (FP RU) ──────────────────────────────────────────

  TEST RESULTS: XGB (FP RU)
  MAE                   : 16.6936
  RMSE                  : 25.9836
  R²                    : 0.8433
  MAPE (%)              : 12.2176
  Max Error             : 108.4740
  train_time_min        : 0.0100

  Subgroup: copolymer_type
  Group                    n      MAE     RMSE       R²
  -----------------------------------------------------
  alternating              8    22.42    30.79   0.7446
  block                    1     4.64     4.64   0.0000
  random                  27    13.14    20.76   0.8129

  Subgroup: stereo_class
  Group                    n      MAE     RMSE       R²
  -----------------------------------------------------
  achiral                  4    34.33    54.89   0.7707
  atactic                  9    18.97    27.43   0.6897
  isotactic               37    11.86    18.73   0.8655
  stereo-irregular         3    21.57    22.01  

In [19]:
m = test_results_reg["xgb_fp_ru"]["clf"] if "clf" in test_results_reg["xgb_fp_ru"] else None
print([ (k, v.get("metrics",{}).get("R²")) for k,v in test_results_reg.items() ])

[('xgb_fp_ru', 0.8432523584364381), ('xgb_fp_poly', 0.8741038685664895)]


In [21]:
import json, numpy as np

with open("test_predictions_reg_full.json") as f:
    pred_abl_data = json.load(f)

print(type(pred_abl_data), 
      list(pred_abl_data.keys()) if isinstance(pred_abl_data, dict) else len(pred_abl_data))
# peek at one entry
k0 = list(pred_abl_data)[0] if isinstance(pred_abl_data, dict) else 0
print("sample entry keys:", pred_abl_data[k0].keys() if isinstance(pred_abl_data[k0], dict) else type(pred_abl_data[k0]))

<class 'dict'> ['gcn_pbsg', 'gcn_smiles', 'gcn_smiles_gl', 'gine_pbsg', 'gine_smiles', 'gine_smiles_gl', 'gatv2_pbsg', 'gatv2_smiles', 'gatv2_smiles_gl', 'rr_fp_pooled_ru', 'rr_fp_ru', 'rr_fp_pooled_poly', 'rr_fp_poly', 'rf_fp_pooled_ru', 'rf_fp_ru', 'rf_fp_pooled_poly', 'rf_fp_poly', 'xgb_fp_pooled_ru', 'xgb_fp_ru', 'xgb_fp_pooled_poly', 'xgb_fp_poly']
sample entry keys: dict_keys(['preds', 'labels'])


In [35]:
with open("test_predictions_reg_ablation.json") as f: abl = json.load(f)
with open("test_predictions_reg_full.json") as f:          full = json.load(f)   # full model, same test set

# pull the arrays for the RU model (match the key names in each file)
y_test   = np.array(abl["xgb_fp_ru"]["labels"])     # true Tm, test set
p_abl    = np.array(abl["xgb_fp_ru"]["preds"])
p_full   = np.array(full["xgb_fp_pooled_ru"]["preds"]) # full model key — adjust name

# map test predictions back to global row indices
test_idx = np.asarray(splits["test"])
# matched rows that are ALSO in test:
matched_rows = sorted({i for idxs in stereo_contrasting.values() for i in idxs})
matched_test = [i for i in matched_rows if i in set(test_idx)]
print(f"matched ∩ test = {len(matched_test)} polymers")

matched ∩ test = 12 polymers


In [36]:
from rdkit import Chem

# group 8 rows — pull from stereo_contrasting (the 8th group)
g8_rows = list(stereo_contrasting.values())[8]
print(f"group 8: {len(g8_rows)} polymers\n")

sub = df.iloc[g8_rows][["repeat_units", "stereo_class", "Pm"]].copy()
sub["Tm"] = y[g8_rows]

# canonicalize each repeat unit so convention differences don't masquerade as chemistry differences
def canon(s):
    m = Chem.MolFromSmiles(str(s))
    return Chem.MolToSmiles(m) if m else f"[unparseable] {s}"
sub["canon"] = sub["repeat_units"].apply(canon)

print(sub.sort_values("Tm").to_string())
print("\nunique canonical repeat units in group:", sub["canon"].nunique())
print("unique stereo classes:", sorted(sub["stereo_class"].unique()))
print(f"Tm range: {sub['Tm'].min():.0f} – {sub['Tm'].max():.0f} °C")

group 8: 27 polymers

          repeat_units      stereo_class    Pm     Tm          canon
172  [*]OC(CC(C)[*])=O  stereo-irregular  0.36   51.0  *OC(=O)CC(*)C
186  [*]OC(CC(C)[*])=O         isotactic  0.88  121.0  *OC(=O)CC(*)C
187  [*]OC(CC(C)[*])=O         isotactic  0.89  122.0  *OC(=O)CC(*)C
165  [*]OC(CC(C)[*])=O      syndiotactic  0.18  126.0  *OC(=O)CC(*)C
164  [*]OC(CC(C)[*])=O  stereo-irregular  0.20  133.0  *OC(=O)CC(*)C
173  [*]OC(CC(C)[*])=O         isotactic  0.91  136.0  *OC(=O)CC(*)C
174  [*]OC(CC(C)[*])=O         isotactic  0.93  143.0  *OC(=O)CC(*)C
175  [*]OC(CC(C)[*])=O         isotactic  0.94  145.0  *OC(=O)CC(*)C
176  [*]OC(CC(C)[*])=O         isotactic  0.93  146.0  *OC(=O)CC(*)C
178  [*]OC(CC(C)[*])=O         isotactic  0.95  147.0  *OC(=O)CC(*)C
177  [*]OC(CC(C)[*])=O         isotactic  0.95  147.0  *OC(=O)CC(*)C
169  [*]OC(CC(C)[*])=O      syndiotactic  0.12  150.0  *OC(=O)CC(*)C
170  [*]OC(CC(C)[*])=O      syndiotactic  0.13  150.0  *OC(=O)CC(*)C
149  [*]OC(C

In [37]:
test_set = set(np.asarray(splits["test"]))
g8_rows = list(stereo_contrasting.values())[8]
g8_test = [i for i in g8_rows if i in test_set]
print(f"{len(g8_test)} of group 8 in test:", g8_test)

# the 5 test members: SMILES, true Tm, stereo
sub5 = df.iloc[g8_test][["repeat_units","stereo_class","Pm"]].copy()
sub5["true_Tm"] = y[g8_test]
print(sub5.sort_values("true_Tm").to_string())

5 of group 8 in test: [164, 171, 174, 176, 180]
          repeat_units      stereo_class    Pm  true_Tm
164  [*]OC(CC(C)[*])=O  stereo-irregular  0.20    133.0
174  [*]OC(CC(C)[*])=O         isotactic  0.93    143.0
176  [*]OC(CC(C)[*])=O         isotactic  0.93    146.0
180  [*]OC(CC(C)[*])=O         isotactic  0.96    156.0
171  [*]OC(CC(C)[*])=O      syndiotactic  0.06    183.0


In [38]:
import json, numpy as np

with open("test_predictions_reg_ablation.json") as f: abl = json.load(f)
with open("test_predictions_reg_full.json") as f:          full = json.load(f)

test_idx = np.asarray(splits["test"])
pos = {gi: j for j, gi in enumerate(test_idx)}
test_set = set(test_idx)

p_abl  = np.array(abl["xgb_fp_ru"]["preds"])
p_full = np.array(full["xgb_fp_pooled_ru"]["preds"])
lab    = np.array(abl["xgb_fp_ru"]["labels"])

print(f"{'grp':>3} {'n':>2} {'nTest':>5} {'trueRng':>7} {'ablRng':>6} {'fullRng':>7} {'ablMAE':>6} {'fullMAE':>7}")
scan = []
for gi, idxs in enumerate(stereo_contrasting.values()):
    idxs = list(idxs)
    tst  = [i for i in idxs if i in test_set]
    true_rng = y[idxs].max() - y[idxs].min()
    if len(tst) >= 2:
        j = [pos[i] for i in tst]
        abl_rng  = p_abl[j].max()  - p_abl[j].min()
        full_rng = p_full[j].max() - p_full[j].min()
        abl_mae  = np.mean(np.abs(y[tst] - p_abl[j]))
        full_mae = np.mean(np.abs(y[tst] - p_full[j]))
        scan.append((gi, len(idxs), len(tst), true_rng, abl_rng, full_rng, abl_mae, full_mae))
        print(f"{gi:>3} {len(idxs):>2} {len(tst):>5} {true_rng:>7.0f} {abl_rng:>6.1f} {full_rng:>7.1f} {abl_mae:>6.1f} {full_mae:>7.1f}")
    else:
        print(f"{gi:>3} {len(idxs):>2} {len(tst):>5} {true_rng:>7.0f}   (<2 test members)")

grp  n nTest trueRng ablRng fullRng ablMAE fullMAE
  0  3     0      29   (<2 test members)
  1  9     1      67   (<2 test members)
  2  8     1      12   (<2 test members)
  3  2     0      31   (<2 test members)
  4 11     1      46   (<2 test members)
  5  5     1     152   (<2 test members)
  6  3     0      76   (<2 test members)
  7  5     0      40   (<2 test members)
  8 27     5     132    0.0    69.4   13.1    15.7
  9  3     0      97   (<2 test members)
 10  2     1      34   (<2 test members)
 11  9     0      54   (<2 test members)
 12  8     1      44   (<2 test members)
 13  5     1      35   (<2 test members)
 14  2     0      22   (<2 test members)


In [39]:
import numpy as np, pandas as pd
rows = []
for gi, idxs in enumerate(stereo_contrasting.values()):
    idxs = list(idxs)
    classes = sorted(df.iloc[idxs]['stereo_class'].unique())
    # confirm single chemistry via canonical SMILES
    from rdkit import Chem
    canon = {Chem.MolToSmiles(Chem.MolFromSmiles(str(s)))
             for s in df.iloc[idxs]['repeat_units']}
    rows.append({
        "set": gi + 1,
        "n_polymers": len(idxs),
        "n_chemistries": len(canon),          # should be 1 for every row
        "stereo_classes": ", ".join(c.replace("stereo-","") for c in classes),
        "Tm_min": round(float(y[idxs].min()), 0),
        "Tm_max": round(float(y[idxs].max()), 0),
        "Tm_range": round(float(y[idxs].max() - y[idxs].min()), 0),
    })
tbl = pd.DataFrame(rows).sort_values("Tm_range", ascending=False)
print(tbl.to_string(index=False))
print(f"\nTOTAL: {tbl.n_polymers.sum()} polymers, {len(tbl)} sets")
print(f"Tm range: median={tbl.Tm_range.median():.0f}  min={tbl.Tm_range.min():.0f}  max={tbl.Tm_range.max():.0f}")
print(f"all single-chemistry: {(tbl.n_chemistries==1).all()}")   # want True

 set  n_polymers  n_chemistries                     stereo_classes  Tm_min  Tm_max  Tm_range
   6           5              1            irregular, syndiotactic    60.0   212.0     152.0
   9          27              1 isotactic, irregular, syndiotactic    51.0   183.0     132.0
  10           3              1            isotactic, syndiotactic   116.0   213.0      97.0
   7           3              1            isotactic, syndiotactic   214.0   291.0      76.0
   2           9              1 isotactic, irregular, syndiotactic   119.0   186.0      67.0
  12           9              1               isotactic, irregular   117.0   171.0      54.0
   5          11              1            irregular, syndiotactic    71.0   117.0      46.0
  13           8              1                 atactic, irregular   225.0   269.0      44.0
   8           5              1            isotactic, syndiotactic   230.0   270.0      40.0
  14           5              1                 atactic, irregular   2